# 🏨 BookingPulse: Hotel Booking Cancellation & Revenue Optimization

**Author:** Venkata Sai Ashrit Kommireddy  

---

## Problem Statement

Hotel cancellations are a significant source of revenue loss. When a booking is cancelled at the last minute, the room often goes unfilled — resulting in direct revenue loss and inefficient resource planning.

This project builds a **cancellation prediction model** using real hotel booking data to:
- Identify which bookings are at high risk of cancellation
- Understand the key drivers behind cancellations
- Provide actionable, data-backed recommendations for hotel revenue teams

**Dataset:** [Hotel Booking Demand Dataset](https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand) — 119,390 bookings across two hotel types (City Hotel and Resort Hotel), covering arrivals from 2015–2017.

---

## Notebook Structure

1. Setup & Data Loading
2. Exploratory Data Analysis (EDA)
3. Feature Engineering & Preprocessing
4. Model Training (XGBoost)
5. Model Evaluation
6. SHAP Explainability
7. Business Insights & Recommendations

## 1. Setup & Data Loading

Installing required libraries and importing dependencies. The dataset (`hotel_bookings.csv`) should be placed in the working directory or uploaded via the Colab file browser.

In [ ]:
!pip install shap xgboost -q

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import shap

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded successfully")

In [ ]:
df = pd.read_csv('hotel_bookings.csv')
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Exploratory Data Analysis (EDA)

Before building any model, we need to understand the data — its shape, missing values, and the patterns that might explain cancellation behavior.

### 2.1 Missing Value Check

In [ ]:
null_summary = df.isnull().sum()
null_summary[null_summary > 0]

Three columns have missing values:
- **`children`** — missing values logically mean 0 children
- **`agent`** — missing means the booking was made directly (no agent), so 0 is appropriate
- **`company`** — same logic; direct bookings have no company ID

We also drop `reservation_status` and `reservation_status_date` — these are direct proxies for cancellation and would cause **data leakage** if kept as features.

In [ ]:
df['children'].fillna(0, inplace=True)
df['agent'].fillna(0, inplace=True)
df['company'].fillna(0, inplace=True)

df.drop(['reservation_status_date', 'reservation_status'], axis=1, inplace=True)

print(f"✅ Cleaned dataset shape: {df.shape}")
print(f"Remaining nulls: {df.isnull().sum().sum()}")

### 2.2 Overall Cancellation Rate

What percentage of bookings are actually cancelled?

In [ ]:
cancel_rate = df['is_canceled'].value_counts(normalize=True)
print("Cancellation Rate:")
print(cancel_rate.rename({0: 'Not Cancelled', 1: 'Cancelled'}).apply(lambda x: f"{x:.1%}"))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
axes[0].pie(
    cancel_rate,
    labels=['Not Cancelled', 'Cancelled'],
    autopct='%1.1f%%',
    colors=['#4CAF50', '#F44336'],
    startangle=90
)
axes[0].set_title('Overall Cancellation Rate', fontsize=13, fontweight='bold')

# By hotel type
sns.countplot(data=df, x='hotel', hue='is_canceled', palette=['#4CAF50', '#F44336'], ax=axes[1])
axes[1].set_title('Cancellations by Hotel Type', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Hotel Type')
axes[1].set_ylabel('Number of Bookings')
axes[1].legend(title='Cancelled', labels=['No', 'Yes'])

plt.tight_layout()
plt.show()

### 2.3 Lead Time vs. Cancellation

**Lead time** is the number of days between the booking date and the arrival date. Intuitively, bookings made far in advance may be more likely to cancel — plans change over time.

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(
    data=df, x='is_canceled', y='lead_time',
    palette=['#4CAF50', '#F44336']
)
plt.title('Lead Time vs. Cancellation', fontsize=13, fontweight='bold')
plt.xlabel('Cancelled (0 = No, 1 = Yes)')
plt.ylabel('Lead Time (days)')
plt.show()

print("Median lead time — Not Cancelled:", df[df['is_canceled']==0]['lead_time'].median(), "days")
print("Median lead time — Cancelled:    ", df[df['is_canceled']==1]['lead_time'].median(), "days")

### 2.4 Country-wise Cancellation Patterns

Looking at the top 10 booking countries — do certain markets cancel more frequently?

In [ ]:
top_countries = df['country'].value_counts().head(10).index
df_top = df[df['country'].isin(top_countries)]

plt.figure(figsize=(13, 5))
sns.countplot(
    data=df_top, x='country', hue='is_canceled',
    order=top_countries, palette=['#4CAF50', '#F44336']
)
plt.title('Top 10 Countries — Cancellation Breakdown', fontsize=13, fontweight='bold')
plt.xlabel('Country')
plt.ylabel('Number of Bookings')
plt.xticks(rotation=45)
plt.legend(title='Cancelled', labels=['No', 'Yes'])
plt.tight_layout()
plt.show()

## 3. Feature Engineering & Preprocessing

To prepare the data for modeling:
- **One-hot encode** all categorical columns using `pd.get_dummies`
- **Engineer `total_guests`** — a derived feature combining adults, children, and babies. Group size can be a useful signal for cancellation behavior.

In [ ]:
df_encoded = pd.get_dummies(df, drop_first=True)

df_encoded['total_guests'] = (
    df_encoded['adults'] +
    df_encoded['children'] +
    df_encoded['babies']
)

print(f"Feature matrix shape: {df_encoded.shape}")
print(f"Total features: {df_encoded.shape[1] - 1} (excluding target)")

In [ ]:
X = df_encoded.drop(['is_canceled'], axis=1)
y = df_encoded['is_canceled']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]:,} rows")
print(f"Test set:     {X_test.shape[0]:,} rows")

## 4. Model Training — XGBoost Classifier

**Why XGBoost?**
- Handles mixed feature types (numeric + one-hot encoded categoricals) well
- Built-in regularization reduces overfitting
- Fast training on tabular data
- Native feature importance — pairs well with SHAP for explainability

We use `eval_metric='logloss'` since this is a binary classification task.

In [ ]:
model = XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("✅ Model trained successfully")

## 5. Model Evaluation

We evaluate on three fronts:
- **Classification Report** — precision, recall, F1 per class
- **ROC-AUC Score** — discrimination ability across thresholds
- **Confusion Matrix** — actual vs. predicted breakdown

In [ ]:
print("=" * 50)
print("Classification Report")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=['Not Cancelled', 'Cancelled']))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred):.4f}")

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Not Cancelled', 'Cancelled'],
    yticklabels=['Not Cancelled', 'Cancelled']
)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title('Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. SHAP Explainability

A high-accuracy model alone isn't enough — hotel revenue managers need to know **why** a booking is flagged as high-risk. SHAP (SHapley Additive exPlanations) provides per-prediction feature attribution, making the model interpretable.

The summary plot below shows:
- **Which features matter most** (ranked by mean |SHAP value|)
- **Direction of impact** — red = high feature value pushes toward cancellation, blue = pushes away

In [ ]:
explainer = shap.Explainer(model)
shap_values = explainer(X_test[:100])

shap.summary_plot(shap_values, X_test[:100], plot_type='dot', show=True)

## 7. Business Insights & Recommendations

Based on the EDA and SHAP analysis, here are the key findings and what hotel revenue teams should do with them:

---

### 📌 Finding 1: Long Lead Time = Higher Cancellation Risk
Cancelled bookings have a significantly higher median lead time than non-cancelled ones. Customers who book far in advance are more likely to change plans.

**Recommendation:** Apply stricter cancellation policies (e.g., non-refundable deposits) for bookings made more than 90 days in advance.

---

### 📌 Finding 2: City Hotels Cancel More Than Resort Hotels
City hotels show a disproportionately higher cancellation rate compared to resort hotels. This likely reflects the business traveler segment, which is more sensitive to itinerary changes.

**Recommendation:** City hotels should consider overbooking strategies calibrated to their historical cancellation rate, similar to airline yield management.

---

### 📌 Finding 3: Repeat Guests Rarely Cancel
Returning customers show significantly lower cancellation rates than first-time bookers.

**Recommendation:** Loyalty programs that incentivize repeat bookings directly reduce cancellation exposure — prioritize these in marketing spend.

---

### 📌 Finding 4: High ADR Bookings Are Higher Risk
Premium-priced bookings (high Average Daily Rate) show elevated cancellation tendency — these are exactly the bookings hotels can least afford to lose.

**Recommendation:** For high-ADR bookings, trigger proactive outreach (personalized confirmations, early check-in offers) 2–4 weeks before arrival to reduce no-shows.

---

### 🚀 What's Next

- **Deploy as a scoring API** — score incoming bookings in real-time and flag high-risk ones for the revenue team dashboard
- **Overbooking model** — use predicted cancellation probability to determine safe overbooking levels per day
- **Pricing integration** — feed risk scores into dynamic pricing to adjust ADR based on cancellation likelihood
- **Segment-level analysis** — break down cancellation drivers by customer segment (OTA vs. direct, corporate vs. leisure)